# Clasificación batch de discurso xenófobo con OpenAI

Esta notebook clasifica textos como `XENOPHOBIC` o `NOT_XENOPHOBIC` usando un modelo base de OpenAI y la Batch API.

In [ ]:
# Si hace falta:
# !pip install -q openai pandas

In [1]:
import os
import json
import time
import random
from typing import Any, Dict, Optional

import pandas as pd
from openai import OpenAI

## Configuración general

In [2]:
MODEL_NAME = 'gpt-4.1-mini'
INPUT_CSV = 'DNU_data/tweets_clasificados.csv'
TEXT_COLUMN = None
MAX_ROWS = None

REQUESTS_JSONL = 'outs_temp/batch_gpt4_xenophobia.jsonl'
RESULTS_JSONL = 'outs_temp/batch_gpt4_xenophobia.jsonl'
OUTPUT_CSV = 'outputs/dnu_gpt_xenofobia.csv'

COMPLETION_WINDOW = '24h'
TEMPERATURE = 0
MAX_TOKENS = 20

# Precios estimativos sincrónicos por 1M tokens.
# Batch suele costar aproximadamente 50% menos.
PRICE_INPUT_PER_1M_SYNC = 0.40
PRICE_OUTPUT_PER_1M_SYNC = 1.60

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
if not os.environ.get('OPENAI_API_KEY'):
    raise ValueError('No encuentro OPENAI_API_KEY en variables de entorno.')

## Cargar dataset

In [3]:
df = pd.read_csv(INPUT_CSV)
print(f'Filas totales: {len(df)}')
print('Columnas:')
print(list(df.columns))
df.head(3)

Filas totales: 259752
Columnas:
['created_at', 'id', 'id_str', 'text', 'source', 'truncated', 'in_reply_to_status_id', 'in_reply_to_status_id_str', 'in_reply_to_user_id', 'quoted_status.place.place_type', 'quoted_status.place.name', 'retweeted_status.quoted_status.quoted_status_id', 'is_retweet', 'caracteres', 'text_pp', 'tokens', 'fecha', 'fecha_dia', 'CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL', 'labels', 'n_labels']


,created_at,id,id_str,text,source,truncated,in_reply_to_status_id,in_reply_to_status_id_str,in_reply_to_user_id,quoted_status.place.place_type,...,WOMEN,LGBTI,RACISM,CLASS,POLITICS,DISABLED,APPEARANCE,CRIMINAL,labels,n_labels
0,Fri Mar 05 11:01:46 +0000 2021,1367792447334580225,1367792447334580225,RT @CELS_Argentina: ✅Celebramos la decisión de...,"<a href=""http://twitter.com/download/android"" ...",False,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,[],0
1,Fri Mar 05 11:06:31 +0000 2021,1367793643239702529,1367793643239702529,RT @LANACION: Ley de Migraciones: el Gobierno ...,"<a href=""http://twitter.com/download/android"" ...",False,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,[],0
2,Fri Mar 05 11:06:33 +0000 2021,1367793651192070144,1367793651192070144,"RT @CRLDEMONIO: Y lo peor , el alto mando d...","<a href=""http://twitter.com/download/android"" ...",False,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,[],0


In [4]:
def detect_text_column(dataframe: pd.DataFrame) -> str:
    candidates = ['text', 'texto', 'tweet', 'contenido', 'content', 'body', 'comment', 'comentario', 'post', 'mensaje', 'message']
    lower_map = {c.lower(): c for c in dataframe.columns}
    for cand in candidates:
        if cand in lower_map:
            return lower_map[cand]
    object_cols = [c for c in dataframe.columns if dataframe[c].dtype == 'object']
    if len(object_cols) == 1:
        return object_cols[0]
    raise ValueError('No pude detectar automáticamente la columna de texto. Define TEXT_COLUMN manualmente.')

text_col = TEXT_COLUMN if TEXT_COLUMN is not None else detect_text_column(df)
work_df = df.copy()
if MAX_ROWS is not None:
    work_df = work_df.head(MAX_ROWS).reset_index(drop=True)

print('Columna de texto:', text_col)
print('Filas a procesar:', len(work_df))

Columna de texto: text
Filas a procesar: 259752


## Instrucción de clasificación

In [5]:
SYSTEM_MESSAGE = '''You are a classifier of xenophobic speech.

Your task is to determine whether a text contains xenophobic hate speech directed at foreigners, immigrants, migrants, or people of a particular nationality or national origin.

Label definitions:
- XENOPHOBIC: the text expresses hostility, contempt, discrimination, dehumanization, exclusion, blame, or hatred against foreigners, immigrants, migrants, or people of a specific nationality or national origin.
- NOT_XENOPHOBIC: the text does not contain xenophobic hate speech.

Return only valid JSON in one line with this exact structure:
{"label":"XENOPHOBIC"}
or
{"label":"NOT_XENOPHOBIC"}'''

USER_TEMPLATE = '{text}'

## Funciones auxiliares

In [6]:
def safe_json_loads(text: Optional[str]) -> Optional[Dict[str, Any]]:
    if text is None:
        return None
    text = str(text).strip()
    try:
        return json.loads(text)
    except Exception:
        start = text.find('{')
        end = text.rfind('}')
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(text[start:end+1])
            except Exception:
                return None
    return None

def extract_label(raw_content: Optional[str]) -> Optional[str]:
    parsed = safe_json_loads(raw_content)
    if isinstance(parsed, dict):
        label = parsed.get('label')
        if label is not None:
            return str(label).strip().upper()
    return None

## Prueba sincrónica rápida sobre 5 casos aleatorios

In [39]:
def test_single_call(text: str) -> Dict[str, Any]:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        messages=[
            {'role': 'system', 'content': SYSTEM_MESSAGE},
            {'role': 'user', 'content': USER_TEMPLATE.format(text=text)}
        ]
    )

    raw = response.choices[0].message.content
    label = extract_label(raw)

    return {
        'raw_response': raw,
        'label': label,
        'prompt_tokens': response.usage.prompt_tokens,
        'completion_tokens': response.usage.completion_tokens,
        'total_tokens': response.usage.total_tokens
    }

SEED = 522
N_SAMPLES = 5
random.seed(SEED)
indices = random.sample(range(len(work_df)), min(N_SAMPLES, len(work_df)))

test_results = []
for i in indices:
    txt = str(work_df[text_col].iloc[i]) if pd.notna(work_df[text_col].iloc[i]) else ''
    out = test_single_call(txt)
    test_results.append(out)

    print('=' * 80)
    print('fila', i)
    print('texto:', repr(txt[:300]))
    print('salida cruda:', repr(out['raw_response']))
    print('label:', out['label'])
    print('tokens:', out['total_tokens'])

fila 20090
texto: 'RT @AXEA65: Si la expulsión es por el incumplimiento del Plan de gobierno y asumir el del candidato perdedor, también deberían ser sanciona…'
salida cruda: '{"label":"NOT_XENOPHOBIC"}'
label: NOT_XENOPHOBIC
tokens: 197
fila 61392
texto: 'Energía que produce México si, no a extranjeros con contratos leoninos'
salida cruda: '{"label":"XENOPHOBIC"}'
label: XENOPHOBIC
tokens: 177
fila 101246
texto: 'RT @maquialifraco: Argentina, el país que exporta jóvenes profesionales e importa delincuentes. https://t.co/dOgwXnOGY8'
salida cruda: '{"label":"XENOPHOBIC"}'
label: XENOPHOBIC
tokens: 195
fila 202958
texto: 'RT @Meruanista: Pedirle a los delincuentes que no hagan destrozos es como pedirle al colegio de profesores que hagan clases'
salida cruda: '{"label":"NOT_XENOPHOBIC"}'
label: NOT_XENOPHOBIC
tokens: 193
fila 224310
texto: '@Hammurabi285639 @hiranechecho @HermogenesPdA Muy bien @camilaFl clarísima !!\nPenas más altas a los delincuentes pa… https://t.co/vTuzXtJAhb'
salida 

## Estimación rápida de costo a partir de la muestra

In [40]:
test_df = pd.DataFrame(test_results)

avg_prompt_tokens = test_df['prompt_tokens'].mean()
avg_completion_tokens = test_df['completion_tokens'].mean()

n = len(work_df)

sync_input_cost = (avg_prompt_tokens * n) / 1_000_000 * PRICE_INPUT_PER_1M_SYNC
sync_output_cost = (avg_completion_tokens * n) / 1_000_000 * PRICE_OUTPUT_PER_1M_SYNC
sync_total = sync_input_cost + sync_output_cost
batch_total = sync_total * 0.5

print('Promedio prompt tokens:', round(avg_prompt_tokens, 2))
print('Promedio completion tokens:', round(avg_completion_tokens, 2))
print('Estimación SINCRÓNICA:', round(sync_total, 2), 'USD')
print('Estimación BATCH:', round(batch_total, 2), 'USD')
print('¿Alcanzan 20 USD?', batch_total <= 20)

Promedio prompt tokens: 184.0
Promedio completion tokens: 10.6
Estimación SINCRÓNICA: 23.52 USD
Estimación BATCH: 11.76 USD
¿Alcanzan 20 USD? True


## Crear archivo JSONL para Batch API

In [42]:
def make_request_line(row_id: int, text: str) -> Dict[str, Any]:
    return {
        'custom_id': f'row-{row_id}',
        'method': 'POST',
        'url': '/v1/chat/completions',
        'body': {
            'model': MODEL_NAME,
            'temperature': TEMPERATURE,
            'max_tokens': MAX_TOKENS,
            'messages': [
                {'role': 'system', 'content': SYSTEM_MESSAGE},
                {'role': 'user', 'content': USER_TEMPLATE.format(text=text)}
            ]
        }
    }

BATCH_SIZE = 20000

num_parts = (len(work_df) + BATCH_SIZE - 1) // BATCH_SIZE
print("Número de batches:", num_parts)

jsonl_files = []

for part in range(num_parts):
    start = part * BATCH_SIZE
    end = min((part + 1) * BATCH_SIZE, len(work_df))

    part_df = work_df.iloc[start:end]

    filename = f"batch_requests_xenophobia_part_{part+1}.jsonl"
    jsonl_files.append(filename)

    with open(filename, 'w', encoding='utf-8') as f:
        for idx, row in part_df.iterrows():
            text = str(row[text_col]) if pd.notna(row[text_col]) else ''
            line = make_request_line(idx, text)
            f.write(json.dumps(line, ensure_ascii=False) + '\n')

    print(f"Archivo creado: {filename} ({len(part_df)} filas)")

Número de batches: 13
Archivo creado: batch_requests_xenophobia_part_1.jsonl (20000 filas)
Archivo creado: batch_requests_xenophobia_part_2.jsonl (20000 filas)
Archivo creado: batch_requests_xenophobia_part_3.jsonl (20000 filas)
Archivo creado: batch_requests_xenophobia_part_4.jsonl (20000 filas)
Archivo creado: batch_requests_xenophobia_part_5.jsonl (20000 filas)
Archivo creado: batch_requests_xenophobia_part_6.jsonl (20000 filas)
Archivo creado: batch_requests_xenophobia_part_7.jsonl (20000 filas)
Archivo creado: batch_requests_xenophobia_part_8.jsonl (20000 filas)
Archivo creado: batch_requests_xenophobia_part_9.jsonl (20000 filas)
Archivo creado: batch_requests_xenophobia_part_10.jsonl (20000 filas)
Archivo creado: batch_requests_xenophobia_part_11.jsonl (20000 filas)
Archivo creado: batch_requests_xenophobia_part_12.jsonl (20000 filas)
Archivo creado: batch_requests_xenophobia_part_13.jsonl (19752 filas)


## Subir archivo y crear el batch

In [30]:
batch_jobs = []

for file in jsonl_files:
    print(f"Subiendo {file}...")

    input_file = client.files.create(
        file=open(file, 'rb'),
        purpose='batch'
    )

    batch = client.batches.create(
        input_file_id=input_file.id,
        endpoint='/v1/chat/completions',
        completion_window=COMPLETION_WINDOW,
        metadata={'job': f'xenophobia-{file}'}
    )

    batch_jobs.append(batch.id)

    print("Batch creado:", batch.id)



Subiendo batch_requests_xenophobia_part_1.jsonl...
Batch creado: batch_69d055d51dac8190b74033f1192fe9b9
Subiendo batch_requests_xenophobia_part_2.jsonl...
Batch creado: batch_69d055ff99ac819092b4cdbc0316d286
Subiendo batch_requests_xenophobia_part_3.jsonl...
Batch creado: batch_69d0562a9bd081908c953e14b591dcf5
Subiendo batch_requests_xenophobia_part_4.jsonl...
Batch creado: batch_69d05655b2b0819089abbed186402091
Subiendo batch_requests_xenophobia_part_5.jsonl...
Batch creado: batch_69d0567f64908190842c44876532c5bb
Subiendo batch_requests_xenophobia_part_6.jsonl...
Batch creado: batch_69d05688d95481909361e6dfc7d7e688


## Consultar estado

In [32]:
POLL_EVERY_SECONDS = 60

remaining = set(batch_jobs)

while remaining:
    for batch_id in list(remaining):
        status = client.batches.retrieve(batch_id)
        print(batch_id, "→", status.status)

        if status.status in {'completed', 'failed', 'expired', 'cancelled'}:
            remaining.remove(batch_id)

    print("Pendientes:", len(remaining))
    time.sleep(POLL_EVERY_SECONDS)

batch_69d055d51dac8190b74033f1192fe9b9 → in_progress
batch_69d0567f64908190842c44876532c5bb → failed
batch_69d05688d95481909361e6dfc7d7e688 → failed
batch_69d0562a9bd081908c953e14b591dcf5 → in_progress
batch_69d05655b2b0819089abbed186402091 → in_progress
batch_69d055ff99ac819092b4cdbc0316d286 → in_progress
Pendientes: 4
batch_69d055d51dac8190b74033f1192fe9b9 → in_progress
batch_69d0562a9bd081908c953e14b591dcf5 → in_progress
batch_69d05655b2b0819089abbed186402091 → in_progress
batch_69d055ff99ac819092b4cdbc0316d286 → in_progress
Pendientes: 4


KeyboardInterrupt: 

In [7]:
failed_batch_ids = [
    "batch_69d0567f64908190842c44876532c5bb",
    "batch_69d05688d95481909361e6dfc7d7e688",
]

for batch_id in failed_batch_ids:
    status = client.batches.retrieve(batch_id)

    print("=" * 100)
    print("BATCH:", batch_id)
    print("status:", status.status)
    print("request_counts:", status.request_counts)

    print("\nerrors:")
    print(status.errors)

    print("\nerror_file_id:")
    print(status.error_file_id)

BATCH: batch_69d0567f64908190842c44876532c5bb
status: failed
request_counts: BatchRequestCounts(completed=0, failed=0, total=0)

errors:
Errors(data=[BatchError(code='token_limit_exceeded', line=None, message='Enqueued token limit reached for gpt-4.1-mini in organization org-1uHjwiaB3OlPzoxfVzhqOSzs. Limit: 40,000,000 enqueued tokens. Please try again once some in_progress batches have been completed.', param=None)], object='list')

error_file_id:
None
BATCH: batch_69d05688d95481909361e6dfc7d7e688
status: failed
request_counts: BatchRequestCounts(completed=0, failed=0, total=0)

errors:
Errors(data=[BatchError(code='token_limit_exceeded', line=None, message='Enqueued token limit reached for gpt-4.1-mini in organization org-1uHjwiaB3OlPzoxfVzhqOSzs. Limit: 40,000,000 enqueued tokens. Please try again once some in_progress batches have been completed.', param=None)], object='list')

error_file_id:
None


## Descargar resultados

In [15]:
if status.status == 'completed':
    output_file_id = status.output_file_id
    print('output_file_id:', output_file_id)

    content = client.files.content(output_file_id)
    with open(RESULTS_JSONL, 'wb') as f:
        f.write(content.read())

    print(f'Resultados guardados en: {RESULTS_JSONL}')
else:
    print('El batch no terminó en estado completed. Revisa errores y archivos de error si existen.')

output_file_id: file-4UsUGLPQZ3XVpkPNY4aoHM
Resultados guardados en: outs_temp/batch_results_xenophobia.jsonl


## Parsear resultados y unirlos al CSV original

In [16]:
VALID_LABELS = {'XENOPHOBIC', 'NOT_XENOPHOBIC'}

rows = []
with open(RESULTS_JSONL, 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        custom_id = obj.get('custom_id')

        raw_content = None
        prompt_tokens = None
        completion_tokens = None
        total_tokens = None
        status_code = None
        error_msg = None

        response = obj.get('response')
        if response:
            status_code = response.get('status_code')
            body = response.get('body', {})
            usage = body.get('usage', {})
            prompt_tokens = usage.get('prompt_tokens')
            completion_tokens = usage.get('completion_tokens')
            total_tokens = usage.get('total_tokens')

            choices = body.get('choices', [])
            if choices and isinstance(choices, list):
                raw_content = choices[0].get('message', {}).get('content')

        if obj.get('error'):
            error_msg = str(obj.get('error'))

        label = extract_label(raw_content)
        invalid_label = None
        if label is not None and label not in VALID_LABELS:
            invalid_label = label

        row_id = int(custom_id.replace('row-', '')) if custom_id and custom_id.startswith('row-') else None

        rows.append({
            'row_id': row_id,
            'batch_raw_response': raw_content,
            'batch_label': label,
            'batch_is_xenophobic': 1 if label == 'XENOPHOBIC' else 0 if label == 'NOT_XENOPHOBIC' else None,
            'batch_invalid_label': invalid_label,
            'batch_prompt_tokens': prompt_tokens,
            'batch_completion_tokens': completion_tokens,
            'batch_total_tokens': total_tokens,
            'batch_status_code': status_code,
            'batch_error': error_msg
        })

res_df = pd.DataFrame(rows).sort_values('row_id')
final_df = work_df.reset_index().rename(columns={'index': 'row_id'}).merge(res_df, on='row_id', how='left')
final_df.to_csv(OUTPUT_CSV, index=False)

print(f'CSV final guardado en: {OUTPUT_CSV}')
final_df.head(10)

CSV final guardado en: outs_temp/muestra_con_batch_gpt_corregido.csv


,row_id,created_at,id,id_str,text,source,truncated,in_reply_to_status_id,in_reply_to_status_id_str,in_reply_to_user_id,...,HATEFUL,batch_raw_response,batch_label,batch_is_xenophobic,batch_invalid_label,batch_prompt_tokens,batch_completion_tokens,batch_total_tokens,batch_status_code,batch_error
0,0,Sat Mar 06 23:16:38 +0000 2021,1368339770686996480,1368339770686996480,Extranjeros participan de los destrozos realiz...,"<a href=""http://twitter.com/download/android"" ...",False,NaN,NaN,NaN,...,0,"{""label"":""NOT_XENOPHOBIC""}",NOT_XENOPHOBIC,0,None,185,11,196,200,None
1,1,Fri Mar 05 17:33:11 +0000 2021,1367890949792272391,1367890949792272391,@pepegoyomr @CARLOSLGUERRERO Al contrario!!! E...,"<a href=""http://twitter.com/download/iphone"" r...",True,1.367887e+18,1.367887e+18,1.420934e+08,...,0,"{""label"":""NOT_XENOPHOBIC""}",NOT_XENOPHOBIC,0,None,194,11,205,200,None
2,2,Fri Mar 05 22:02:06 +0000 2021,1367958627739258880,1367958627739258880,La diferencia con los kirchneristas ya no es s...,"<a href=""http://twitter.com/download/iphone"" r...",True,NaN,NaN,NaN,...,0,"{""label"":""NOT_XENOPHOBIC""}",NOT_XENOPHOBIC,0,None,188,11,199,200,None
3,3,Sun Mar 07 14:26:52 +0000 2021,1368568840351870976,1368568840351870976,@Lady_Chiyome La dictadura de pelusones del 73...,"<a href=""https://mobile.twitter.com"" rel=""nofo...",True,1.368373e+18,1.368373e+18,1.360764e+18,...,0,"{""label"":""NOT_XENOPHOBIC""}",NOT_XENOPHOBIC,0,None,194,11,205,200,None
4,4,Sun Mar 07 02:07:27 +0000 2021,1368382756502306816,1368382756502306816,@Mari21Llll La pinché Momia solo es un florero...,"<a href=""http://twitter.com/download/iphone"" r...",True,1.368354e+18,1.368354e+18,1.274648e+18,...,0,"{""label"":""NOT_XENOPHOBIC""}",NOT_XENOPHOBIC,0,None,196,11,207,200,None
5,5,Fri Mar 05 11:33:53 +0000 2021,1367800528059707392,1367800528059707392,El Gobierno derogó el decreto de Mauricio Macr...,"<a href=""https://zapier.com/"" rel=""nofollow"">Z...",True,NaN,NaN,NaN,...,0,"{""label"":""NOT_XENOPHOBIC""}",NOT_XENOPHOBIC,0,None,187,11,198,200,None
6,6,Sat Mar 06 12:45:50 +0000 2021,1368181023754887170,1368181023754887170,@MikeHam060 Fueron expulsados por tener antece...,"<a href=""https://mobile.twitter.com"" rel=""nofo...",True,1.368175e+18,1.368175e+18,1.102314e+18,...,0,"{""label"":""NOT_XENOPHOBIC""}",NOT_XENOPHOBIC,0,None,188,11,199,200,None
7,7,Fri Mar 05 19:20:32 +0000 2021,1367917965572775943,1367917965572775943,@bastogne1943 Exacto... Hoy los únicos capaces...,"<a href=""http://twitter.com/download/android"" ...",True,1.367881e+18,1.367881e+18,1.072829e+18,...,0,"{""label"":""NOT_XENOPHOBIC""}",NOT_XENOPHOBIC,0,None,195,11,206,200,None
8,8,Fri Mar 05 17:18:08 +0000 2021,1367887162449002504,1367887162449002504,@rubenprofe @Elputonisa @ArielSanche2 @Dasmatn...,"<a href=""https://mobile.twitter.com"" rel=""nofo...",True,1.367887e+18,1.367887e+18,1.500477e+08,...,0,"{""label"":""NOT_XENOPHOBIC""}",NOT_XENOPHOBIC,0,None,203,11,214,200,None
9,9,Fri Mar 05 23:28:20 +0000 2021,1367980325494734849,1367980325494734849,@neuquen24horas @alferdez @alferdezprensa siem...,"<a href=""http://twitter.com/download/iphone"" r...",True,1.367850e+18,1.367850e+18,7.575446e+08,...,0,"{""label"":""NOT_XENOPHOBIC""}",NOT_XENOPHOBIC,0,None,197,11,208,200,None


## Resumen rápido

In [17]:
print('Casos clasificados como XENOPHOBIC:', int((final_df['batch_label'] == 'XENOPHOBIC').sum()))
print('Casos clasificados como NOT_XENOPHOBIC:', int((final_df['batch_label'] == 'NOT_XENOPHOBIC').sum()))
print('Etiquetas inválidas:', int(final_df['batch_invalid_label'].notna().sum()))
print('Distribución:')
print(final_df['batch_label'].value_counts(dropna=False))

Casos clasificados como XENOPHOBIC: 22
Casos clasificados como NOT_XENOPHOBIC: 78
Etiquetas inválidas: 0
Distribución:
batch_label
NOT_XENOPHOBIC    78
XENOPHOBIC        22
Name: count, dtype: int64


## Costo observado real del batch

In [18]:
prompt_sum = pd.to_numeric(final_df['batch_prompt_tokens'], errors='coerce').fillna(0).sum()
completion_sum = pd.to_numeric(final_df['batch_completion_tokens'], errors='coerce').fillna(0).sum()

sync_input_cost_real = prompt_sum / 1_000_000 * PRICE_INPUT_PER_1M_SYNC
sync_output_cost_real = completion_sum / 1_000_000 * PRICE_OUTPUT_PER_1M_SYNC
batch_cost_real = (sync_input_cost_real + sync_output_cost_real) * 0.5

print('Prompt tokens:', int(prompt_sum))
print('Completion tokens:', int(completion_sum))
print('Costo batch estimado real:', round(batch_cost_real, 4), 'USD')

Prompt tokens: 18629
Completion tokens: 1078
Costo batch estimado real: 0.0046 USD


## Reintento de batches fallidos

Los batches de las partes 5 y 6 fallaron por `token_limit_exceeded` (cola de 40M tokens ocupada).
Ahora que los batches anteriores estan `completed`, se reenvian solo esas dos partes.
Los archivos JSONL ya existen en disco; si no estuvieran, la celda los recrea.

In [ ]:
# Partes que fallaron
FAILED_PARTS = [5, 6]
POLL_EVERY_SECONDS = 60

# 1. Recrear JSONL si no existe
import os
for part in FAILED_PARTS:
    filename = f"batch_requests_xenophobia_part_{part}.jsonl"
    if not os.path.exists(filename):
        print(f"Recreando {filename}...")
        start = (part - 1) * BATCH_SIZE
        end   = min(part * BATCH_SIZE, len(work_df))
        part_df = work_df.iloc[start:end]
        with open(filename, "w", encoding="utf-8") as fout:
            for idx, row in part_df.iterrows():
                text = str(row[text_col]) if pd.notna(row[text_col]) else ""
                fout.write(json.dumps(make_request_line(idx, text), ensure_ascii=False) + "\n")
        print(f"  {len(part_df)} filas escritas.")
    else:
        print(f"OK {filename} ya existe.")

# 2. Subir y crear nuevos batches
retry_jobs = {}
for part in FAILED_PARTS:
    filename = f"batch_requests_xenophobia_part_{part}.jsonl"
    print(f"Subiendo {filename}...")
    input_file = client.files.create(
        file=open(filename, "rb"),
        purpose="batch"
    )
    batch = client.batches.create(
        input_file_id=input_file.id,
        endpoint="/v1/chat/completions",
        completion_window=COMPLETION_WINDOW,
        metadata={"job": f"xenophobia-retry-part{part}"}
    )
    retry_jobs[part] = batch.id
    print(f"  Batch creado: {batch.id}")

print("Batches en retry:", retry_jobs)

# 3. Polling
remaining = set(retry_jobs.values())
part_by_id = {v: k for k, v in retry_jobs.items()}
completed_batches = {}
while remaining:
    for batch_id in list(remaining):
        st = client.batches.retrieve(batch_id)
        print(time.strftime("%H:%M:%S"), f"parte {part_by_id[batch_id]}", "->", st.status)
        if st.status in {"completed", "failed", "expired", "cancelled"}:
            completed_batches[batch_id] = st
            remaining.remove(batch_id)
    if remaining:
        time.sleep(POLL_EVERY_SECONDS)

# 4. Descargar y parsear resultados
retry_rows = []
for batch_id, st in completed_batches.items():
    part = part_by_id[batch_id]
    print(f"Parte {part} -> {st.status}")
    if st.status != "completed":
        print(f"  SKIP: {st.errors}")
        continue
    result_file = f"outs_temp/batch_results_xenophobia_part_{part}.jsonl"
    content = client.files.content(st.output_file_id)
    with open(result_file, "wb") as fout:
        fout.write(content.read())
    print(f"  Guardado en {result_file}")
    with open(result_file, "r", encoding="utf-8") as fin:
        for line in fin:
            obj = json.loads(line)
            custom_id = obj.get("custom_id")
            raw_content = None
            prompt_tokens = completion_tokens = total_tokens = status_code = None
            error_msg = None
            response = obj.get("response")
            if response:
                status_code = response.get("status_code")
                body = response.get("body", {})
                usage = body.get("usage", {})
                prompt_tokens = usage.get("prompt_tokens")
                completion_tokens = usage.get("completion_tokens")
                total_tokens = usage.get("total_tokens")
                choices = body.get("choices", [])
                if choices:
                    raw_content = choices[0].get("message", {}).get("content")
            if obj.get("error"):
                error_msg = str(obj.get("error"))
            label = extract_label(raw_content)
            row_id = int(custom_id.replace("row-", "")) if custom_id and custom_id.startswith("row-") else None
            retry_rows.append({
                "row_id": row_id,
                "batch_raw_response": raw_content,
                "batch_label": label,
                "batch_is_xenophobic": 1 if label == "XENOPHOBIC" else 0 if label == "NOT_XENOPHOBIC" else None,
                "batch_invalid_label": label if label not in {"XENOPHOBIC", "NOT_XENOPHOBIC", None} else None,
                "batch_prompt_tokens": prompt_tokens,
                "batch_completion_tokens": completion_tokens,
                "batch_total_tokens": total_tokens,
                "batch_status_code": status_code,
                "batch_error": error_msg
            })

retry_df = pd.DataFrame(retry_rows).sort_values("row_id").reset_index(drop=True)
print(f"Filas recuperadas: {len(retry_df)}")
print(retry_df["batch_label"].value_counts(dropna=False))
